In [1]:
import pandas as pd
from docx import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama

print("Setup successful")

C:\Users\dccha\AppData\Local\Temp\ipykernel_35780\2629374278.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
C:\Users\dccha\OneDrive\Desktop\HR_Chatbot\hr_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup successful


In [2]:
llm = ChatOllama(
    model="qwen2.5:1.5b"
)

In [3]:
response = llm.invoke("Say hello")

print(response.content)

Hello! How can I help you today?


In [4]:
excel_file = "data/HR_Data.xlsx"

employee_df = pd.read_excel(
    excel_file,
    sheet_name="Employee_Master_Data"
)

attendance_df = pd.read_excel(
    excel_file,
    sheet_name="Attendance"
)

timesheet_df = pd.read_excel(
    excel_file,
    sheet_name="Timesheet"
)

leave_df = pd.read_excel(
    excel_file,
    sheet_name="Leave_Tracker"
)

In [5]:
from docx import Document

doc = Document("data/Employee_Handbook.docx")

text = ""

for para in doc.paragraphs:
    text += para.text + "\n"

In [6]:
from langchain_core.documents import Document as LangDocument

documents = [
    LangDocument(page_content=text)
]

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

In [8]:
len(chunks)

6

In [9]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2529.19it/s]


In [10]:
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

In [11]:
vectorstore.save_local("faiss_index")

In [12]:
docs = vectorstore.similarity_search(
    "What is the leave policy?",
    k=3
)

for doc in docs:
    print(doc.page_content)
    print("="*10)

8. EXIT & OFFBOARDING
Notice period: 90 days post confirmation. On exit, you'll get:
  • Final salary, leave encashment, and gratuity (if applicable)
  • Relieving letter and experience certificate
  • Form 16 by 15 June following the FY

Return Company laptop, ID card, and other assets on the last working day.

9. AMENDMENTS
We update this handbook as needed. Material changes are emailed to all employees and require fresh acknowledgement.
2. WORKING WITH US
  • Hours: [11:00 AM – 08:00 PM, Mon–Fri]. 
  • Holidays: [12] national + festival holidays. List published in January.
  • Leave: [20 EL + 12 CL + 10 SL]. Apply via the HR portal at least [3] days in advance.
  • Pay day: Salary is credited by last day of every month.

3. BENEFITS
  • Group medical insurance (₹1 cover, employee + 5 dependents)
  • Term life and accident cover
  • [optional: ESOPs, learning budget, gym, meal cards]
ACKNOWLEDGEMENT
I have received and read the PyApp Technologies Pvt. Ltd.   Employee Handbook v1.0. I

In [13]:
def handbook_query(question):

    docs = vectorstore.similarity_search(
        question,
        k=3
    )

    context = "\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
You are an HR assistant for a startup.

Answer ONLY from the provided context.
If the answer is not present in the context, say:
"I could not find that information in the HR handbook."

HR Handbook Context:
{context}

Question:
{question}
"""

    response = llm.invoke(prompt)

    return response.content

In [14]:
print(handbook_query("What is the leave policy?"))

The leave policy allows 20 days of annual leave (EL), 12 days of compensatory time off (CL), and 10 days of sick leave (SL). These can be applied for at least three days in advance.


In [15]:
def get_manager(employee_name):
    result = employee_df[
        employee_df["name"].str.lower() == employee_name.lower()
    ]

    if result.empty:
        return "Employee not found."

    return f"{employee_name}'s manager is {result.iloc[0]['manager']}."

In [16]:
def get_employee_details(employee_name, query):

    result = employee_df[
        employee_df["name"].str.lower() == employee_name.lower()
    ]

    if result.empty:
        return "Employee not found."

    row = result.iloc[0]


    if "department" in query or "dept" in query:
        return f"{employee_name}'s department is {row['department']}."


    elif "joining" in query or "join date" in query or "joining_date" in query:
        return f"{employee_name} joined on {row['joining_date']}."


    return "Information not available."

In [17]:
def get_leave_info(employee_name, query):

    employee = leave_df[
        leave_df["Name"].str.lower() == employee_name.lower()
    ]

    if employee.empty:
        return "Employee not found."

    row = employee.iloc[0]


    # Specific CL used
    if "cl" in query and any(word in query for word in ["used", "taken", "utilized", "consumed"]):
        return f"{employee_name}'s CL used count is {row['CL Taken']} days."


    # Specific EL used
    elif "el" in query and any(word in query for word in ["used", "taken", "utilized", "consumed"]):
        return f"{employee_name}'s EL used count is {row['EL Taken']} days."


    # Specific SL used
    elif "sl" in query and any(word in query for word in ["used", "taken", "utilized", "consumed"]):
        return f"{employee_name}'s SL used count is {row['SL Taken']} days."


    # Total used leaves
    elif any(word in query for word in ["used", "taken", "utilized", "consumed"]):
        return f"{employee_name} has used {row['Total Taken']} total leaves."


    # Specific CL balance
    elif "cl" in query and any(word in query for word in ["balance", "left", "remaining"]):
        return f"{employee_name}'s CL balance is {row['CL Balance']} days."


    # Specific EL balance
    elif "el" in query and any(word in query for word in ["balance", "left", "remaining"]):
        return f"{employee_name}'s EL balance is {row['EL Balance']} days."


    # Specific SL balance
    elif "sl" in query and any(word in query for word in ["balance", "left", "remaining"]):
        return f"{employee_name}'s SL balance is {row['SL Balance']} days."


    # Complete leave details
    else:
        return f"""
{employee_name}'s Leave Details:

CL Taken: {row['CL Taken']}
EL Taken: {row['EL Taken']}
SL Taken: {row['SL Taken']}

CL Balance: {row['CL Balance']}
EL Balance: {row['EL Balance']}
SL Balance: {row['SL Balance']}

Total Taken: {row['Total Taken']}
Total Balance: {row['Total Balance']}
"""

In [18]:
def get_attendance(employee_name):
    result = attendance_df[
        attendance_df["Name"].str.lower() == employee_name.lower()
    ]

    if result.empty:
        return "Employee not found."

    row = result.iloc[0]

    return f"""
Attendance Percentage: {row['Attendance %']}%
Present Days: {row['Present']}
Paid Leave: {row['Paid Leave']}
Absent Days: {row['Absent (LOP)']}
"""

In [19]:
def get_attendance_details(employee_name, query):

    result = attendance_df[
        attendance_df["Name"].str.lower() == employee_name.lower()
    ]

    if result.empty:
        return "Employee not found."

    row = result.iloc[0]

    absent_days = []
    leave_days = []
    present_days = []
    weekoff_days = []

    for day in range(1, 31):

        status = str(row[str(day)]).strip().upper()

        if status == "A":
            absent_days.append(day)

        elif status == "L":
            leave_days.append(day)

        elif status == "P":
            present_days.append(day)

        elif status == "WO":
            weekoff_days.append(day)


    # Specific absent days
    if "absent" in query or "absence" in query:
        return f"{employee_name} was absent on days: {absent_days}"


    # Specific leave days
    elif "leave" in query:
        return f"{employee_name} was on leave on days: {leave_days}"


    # Specific week off
    elif "week off" in query or "weekoff" in query or "wo" in query:
        return f"{employee_name}'s week off days are: {weekoff_days}"


    # Complete attendance details
    else:
        return f"""
{employee_name}'s Attendance Details:

Present Days: {len(present_days)}
Absent Days: {len(absent_days)}
Leave Days: {len(leave_days)}
Week Off Days: {len(weekoff_days)}

Absent on: {absent_days}
Leave on: {leave_days}
Week Off on: {weekoff_days}
"""

In [20]:
def get_attendance_summary(employee_name, query):

    result = attendance_df[
        attendance_df["Name"].str.lower() == employee_name.lower()
    ]

    if result.empty:
        return "Employee not found."

    row = result.iloc[0]


    if "payable" in query:
        return f"{employee_name}'s payable days are {row['Payable Days']}."


    elif "attendance" in query or "attendance %" in query:
        return f"{employee_name}'s attendance percentage is {row['Attendance %']}%."


    return "Attendance information not available."

In [21]:
def hr_bot(question):

    q = question.lower()

    # Manager query
    if "manager" in q:

        for name in employee_df["name"]:
            if name.lower() in q:
                return get_manager(name)


   # Leave query
    if any(word in q for word in [
        "cl",
        "el",
        "sl",
        "leave",
        "balance",
        "used",
        "taken",
        "utilized",
        "consumed",
        "remaining",
        "left"
    ]):

        for name in leave_df["Name"]:
            if name.lower() in q:

                return get_leave_info(name, q)


    # Attendance query
    if any(word in q for word in [
        "attendance",
        "present",
        "absent",
        "absence",
        "lop",
        "week off",
        "weekoff",
        "wo"
    ]):

        for name in attendance_df["Name"]:
            if name.lower() in q:
                return get_attendance_details(name, q)

    # Employee details query
    if any(word in q for word in [
        "department",
        "dept",
        "joining",
        "join date"
    ]):

        for name in employee_df["name"]:
            if name.lower() in q:
                return get_employee_details(name, q)



    # Attendance summary query
    if any(word in q for word in [
        "payable",
        "attendance %"
    ]):

        for name in attendance_df["Name"]:
            if name.lower() in q:
                return get_attendance_summary(name, q)


    # Handbook RAG
    return handbook_query(question)

In [29]:
while True:
    question = input("Ask HR Bot: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = hr_bot(question)
    print("\nHR Bot:", answer)
    print()

Ask HR Bot:  what is hr policy?



HR Bot: I could not find that information in the HR handbook.



Ask HR Bot:  what is the company name?



HR Bot: The company name is PyApp Technologies Pvt. Ltd.



Ask HR Bot:  how many days of notice period?



HR Bot: 90 days post confirmation.



... (4 outputs left out)

Ask HR Bot:  ok exit



HR Bot: You will get your final salary, leave encashment, and gratuity (if applicable) at the end of 90 days post confirmation. You will also receive a relieving letter and experience certificate on the last working day. Additionally, you are required to return your company laptop, ID card, and any other assets by the last working day.



Ask HR Bot:  exit


Goodbye!


In [27]:
print(employee_df["name"].tolist())

['Amit Singh', 'Ankit Singh', 'Ravi', 'Mohit', 'Sapna']


In [23]:
print(type(vectorstore))

<class 'langchain_community.vectorstores.faiss.FAISS'>


In [24]:
print(type(llm))

<class 'langchain_ollama.chat_models.ChatOllama'>


In [25]:
print(type(employee_df))

<class 'pandas.DataFrame'>


In [26]:
print(type(chunks))

<class 'list'>
